In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ajayvishwanatha/yelp-review/business_data_cleaned.csv
/kaggle/input/datasets/ajayvishwanatha/yelp-review/review_train_y.csv
/kaggle/input/datasets/ajayvishwanatha/yelp-review/user_clean.csv
/kaggle/input/datasets/ajayvishwanatha/yelp-review/review_train_x.csv
/kaggle/input/datasets/ajayvishwanatha/yelp-review/review_test_x.csv
/kaggle/input/datasets/ajayvishwanatha/yelp-review/tips_cleaned.csv


In [2]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, log_loss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore")

# Global Config

In [3]:
business = pd.read_csv('/kaggle/input/datasets/ajayvishwanatha/yelp-review/business_data_cleaned.csv')
user = pd.read_csv('/kaggle/input/datasets/ajayvishwanatha/yelp-review/user_clean.csv')
tip = pd.read_csv('/kaggle/input/datasets/ajayvishwanatha/yelp-review/tips_cleaned.csv')

In [4]:
# Loading train and test datasets:
test_x = pd.read_csv('/kaggle/input/datasets/ajayvishwanatha/yelp-review/review_test_x.csv')
train_x = pd.read_csv('/kaggle/input/datasets/ajayvishwanatha/yelp-review/review_train_x.csv')
train_y = pd.read_csv('/kaggle/input/datasets/ajayvishwanatha/yelp-review/review_train_y.csv')

Column Rename

In [5]:
user = user.rename(columns={
    "review_count": "user_review_count",
    "fans": "user_fans",
    "average_stars": "user_avg_stars",
    "friends_count": "user_friends_count",
    "is_elite": "user_is_elite",
    "elite_years": "user_elite_years"
})

In [6]:
business = business.rename(columns={
    "review_count": "business_review_count",
    "stars": "business_stars",
    "is_open": "business_is_open"
})

In [7]:
tip = tip.rename(columns={
    "compliment_count": "tip_compliment",
    "text": "tip_text",
    "date": "tip_date"
})

Feature Engineering

In [8]:
null_columns = ['BusinessAcceptsCreditCards','BikeParking','ByAppointmentOnly','RestaurantsReservations','HasTV','Caters',
                'RestaurantsTakeOut','HappyHour','OutdoorSeating','RestaurantsDelivery','BusinessAcceptsBitcoin',
                'WheelchairAccessible','DogsAllowed','Corkage','DriveThru','CoatCheck','BYOB','BusinessParking',
                'GoodForMeal','Ambience','Music','HairSpecializesIn','BestNights','RestaurantsAttire','Alcohol',
                'AgesAllowed','BYOBCorkage','Smoking','WiFi','NoiseLevel']

In [9]:
business[null_columns] = business[null_columns].fillna('Unknown')

In [10]:
business[null_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype 
---  ------                      --------------  ----- 
 0   BusinessAcceptsCreditCards  15000 non-null  object
 1   BikeParking                 15000 non-null  object
 2   ByAppointmentOnly           15000 non-null  object
 3   RestaurantsReservations     15000 non-null  object
 4   HasTV                       15000 non-null  object
 5   Caters                      15000 non-null  object
 6   RestaurantsTakeOut          15000 non-null  object
 7   HappyHour                   15000 non-null  object
 8   OutdoorSeating              15000 non-null  object
 9   RestaurantsDelivery         15000 non-null  object
 10  BusinessAcceptsBitcoin      15000 non-null  object
 11  WheelchairAccessible        15000 non-null  object
 12  DogsAllowed                 15000 non-null  object
 13  Corkage                     15000 non-null  ob

In [11]:
business['WiFi'] =business['WiFi'].replace('Paid','paid')

In [12]:
bool_cols = business.select_dtypes(include=['bool']).columns

In [13]:
business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Columns: 112 entries, business_id to is_hours_missing
dtypes: bool(56), float64(3), int64(15), object(38)
memory usage: 7.2+ MB


In [14]:
object_cols= business.select_dtypes(include=['object']).columns

In [15]:
cat_cols=['BusinessAcceptsCreditCards', 'BikeParking',
       'ByAppointmentOnly', 'RestaurantsReservations', 'RestaurantsAttire',
       'Alcohol', 'HasTV', 'Caters', 'RestaurantsTakeOut', 'NoiseLevel',
       'HappyHour', 'OutdoorSeating', 'WiFi', 'RestaurantsDelivery',
       'BusinessAcceptsBitcoin', 'WheelchairAccessible', 'DogsAllowed',
       'Corkage', 'DriveThru', 'CoatCheck', 'Smoking', 'BYOB', 'BYOBCorkage',
       'BusinessParking', 'GoodForMeal', 'AgesAllowed', 'Ambience', 'Music',
       'HairSpecializesIn', 'BestNights']

In [16]:
business[cat_cols] = business[cat_cols].astype('category')

In [17]:
business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Columns: 112 entries, business_id to is_hours_missing
dtypes: bool(56), category(30), float64(3), int64(15), object(8)
memory usage: 4.2+ MB


user feature engineering

In [18]:
user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 992656 entries, 0 to 992655
Data columns (total 13 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   user_id             992656 non-null  object 
 1   user_review_count   992656 non-null  int64  
 2   yelping_since       992656 non-null  object 
 3   useful              992656 non-null  int64  
 4   elite               71540 non-null   object 
 5   friends             609371 non-null  object 
 6   user_fans           992656 non-null  int64  
 7   user_avg_stars      992656 non-null  float64
 8   elite_na_flag       992656 non-null  int64  
 9   friends_na_flag     992656 non-null  int64  
 10  user_is_elite       992656 non-null  int64  
 11  user_elite_years    992656 non-null  int64  
 12  user_friends_count  992656 non-null  int64  
dtypes: float64(1), int64(8), object(4)
memory usage: 98.5+ MB


In [19]:
user.isna().sum()

user_id                    0
user_review_count          0
yelping_since              0
useful                     0
elite                 921116
friends               383285
user_fans                  0
user_avg_stars             0
elite_na_flag              0
friends_na_flag            0
user_is_elite              0
user_elite_years           0
user_friends_count         0
dtype: int64

In [20]:
user['elite'].value_counts()

elite
2019,20,20,2021              6195
2018,2019,20,20,2021         5335
2021                         4037
2017,2018,2019,20,20,2021    4003
20,20,2021                   3782
                             ... 
2014,2017,2018,20,20,2021       1
2013,2014,2015,2017,2019        1
2007,2019                       1
2010,2014,2015,2016             1
2008,2014,2015,2016             1
Name: count, Length: 1240, dtype: int64

In [21]:
user['elite_year_count'] = user['elite'].apply(lambda x: len(str(x).split(',')) if str(x).strip() not in ('', 'nan', 'None', 'NaT') else 0)

In [22]:
user['elite_year_count'].value_counts()

elite_year_count
0     921116
3      11222
2      11066
4      10773
1      10606
5       8422
6       6012
7       4172
8       3183
9       1665
11      1305
10      1211
12       716
13       578
14       342
15       149
16        78
17        40
Name: count, dtype: int64

In [23]:
bins = [0, 1, 3, 6, 10, float('inf')]
labels = ['Not Elite', '1-2 Years', '3-5 Years', '6-9 Years', '10+ Years']

user['elite_year_bin'] = pd.cut(user['elite_year_count'], bins=bins, labels=labels, right=False)

Tip Feature engineering

In [24]:
tip.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300845 entries, 0 to 300844
Data columns (total 7 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   user_id               300845 non-null  object
 1   business_id           300845 non-null  object
 2   tip_text              300841 non-null  object
 3   tip_date              300845 non-null  object
 4   tip_compliment        300845 non-null  int64 
 5   text_is_null_tips     300845 non-null  bool  
 6   duplicated_rows_tips  300845 non-null  bool  
dtypes: bool(2), int64(1), object(4)
memory usage: 12.1+ MB


merging the dataframe

In [25]:
train_x["is_train"] = 1
test_x["is_train"] = 0

In [26]:
combined = pd.concat([train_x,test_x], axis=0).reset_index(drop=True)

In [27]:
combined = combined.merge(user, on="user_id", how="left")
combined = combined.merge(business, on="business_id", how="left")

In [28]:
tip_aggregate = tip.groupby('business_id').agg(
    tip_count           = ('tip_text', 'count'),      
    tip_avg_compliments = ('tip_compliment', 'mean')).reset_index()

print(f"tip_aggregate shape: {tip_aggregate.shape}")
print(tip_aggregate.head())

tip_aggregate shape: (14686, 3)
        business_id  tip_count  tip_avg_compliments
0  0000a2efa84a6d8e          8             0.000000
1  0001eb825604ba80         10             0.000000
2  0003118d1ad3ffcd         24             0.083333
3  0005dc2447718e1b          1             0.000000
4  000cee8e368586cf         12             0.000000


In [29]:
combined = combined.merge(tip_aggregate, on='business_id', how='left')

# Fill NaNs (businesses with no tips)
combined[['tip_count', 'tip_avg_compliments']] = combined[['tip_count', 'tip_avg_compliments']].fillna(0)

# Verify
print(combined[['tip_count', 'tip_avg_compliments']].isna().sum())

tip_count              0
tip_avg_compliments    0
dtype: int64


In [30]:
bins = [0, 100, 250, 500, 1000, float('inf')]
labels = ['very_short', 'short', 'medium', 'long', 'very_long']

combined['text_len_bin'] = pd.cut(combined['text'].str.len(), bins=bins, labels=labels, include_lowest=True)

adding time related feature engineering

In [31]:
combined['date'] = pd.to_datetime(combined['date'])
combined['yelping_since'] = pd.to_datetime(combined['yelping_since'])

combined['review_year'] = combined['date'].dt.year
combined['review_month'] = combined['date'].dt.month
combined['day_of_week'] = combined['date'].dt.dayofweek
combined['is_weekend'] = (combined['day_of_week'] >= 5).astype(int)

combined['month_sin'] = np.sin(2 * np.pi * combined['review_month'] / 12)
combined['month_cos'] = np.cos(2 * np.pi * combined['review_month'] / 12)
combined['day_sin'] = np.sin(2 * np.pi * combined['day_of_week'] / 7)
combined['day_cos'] = np.cos(2 * np.pi * combined['day_of_week'] / 7)

combined['user_tenure_at_review'] = (combined['date'] - combined['yelping_since']).dt.days
max_date = combined['date'].max()
combined['review_age_days'] = (max_date - combined['date']).dt.days

combined['star_diff'] = combined['stars'] - combined['business_stars']
combined['is_elite_at_review_time'] = combined.apply(lambda x: str(x['review_year']) in str(x['elite']), axis=1).astype(int)

In [32]:
train_x = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_x  = combined[combined['is_train'] == 0].drop(columns=['is_train'])

Merging Train_x and Train_Y

In [33]:
train = pd.concat([train_x,train_y],axis=1)

In [34]:
train.head()

,review_id,user_id,business_id,stars,text,date,user_review_count,yelping_since,useful,elite,...,is_weekend,month_sin,month_cos,day_sin,day_cos,user_tenure_at_review,review_age_days,star_diff,is_elite_at_review_time,top_useful
0,a021fc6e6f9f0a89,cc59fcfd2eef6f33,3246f2e18cd91125,10.00,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,16.0,2014-01-17 19:20:57,1.0,NaN,...,1,5.000000e-01,8.660254e-01,-0.781831,0.623490,351.0,2572,3.440,0,0
1,e2f8f0b977247525,887f36457cda93f3,9b667f612347f52d,1.04,I am a long term frequent customer of this est...,2015-09-23 23:10:31,6.0,2014-06-15 18:44:27,5.0,NaN,...,0,-1.000000e+00,-1.836970e-16,0.974928,-0.222521,465.0,2309,-5.520,0,0
2,38306a152b067345,40427c5f2fe0262d,4a305623b48d793d,6.56,Love going here for happy hour or dinner! Gre...,2014-06-27 22:44:01,272.0,2011-05-15 22:03:03,112.0,"2014,2015,2016",...,0,1.224647e-16,-1.000000e+00,-0.433884,-0.900969,1139.0,2762,1.345,1,0
3,9401208079837bdf,6a8b86249ddfe6eb,583a05bb24b08856,10.00,My absolute favorite cafe in the city. Their b...,2014-11-12 15:30:27,362.0,2010-02-22 16:15:28,217.0,2018,...,0,-5.000000e-01,8.660254e-01,0.974928,-0.222521,1723.0,2625,3.440,0,0
4,4528c75b77b05a8e,f8a19bf5830cb4a8,e08f9589bffb26f7,10.00,HOLY SMOKES!\n\nactual pumpkin pie mixed in wi...,2009-10-13 19:49:51,604.0,2006-07-13 19:18:52,504.0,"2007,2008,2009,2010,2011",...,0,-8.660254e-01,5.000000e-01,0.781831,0.623490,1188.0,4480,1.855,1,0


In [41]:
cols_to_drop = ['review_id', 'user_id', 'business_id', 'text', 'date', 
                'yelping_since', 'elite', 'categories']

In [42]:
y = train['top_useful'].values

In [43]:
X_train = train.drop(columns=cols_to_drop + ['top_useful'])
X_test = test_x.drop(columns=cols_to_drop)

In [44]:
X_train['city'] = X_train['city'].astype('category')
X_train['state'] = X_train['state'].astype('category')
X_test['city'] = X_test['city'].astype('category')
X_test['state'] = X_test['state'].astype('category')

In [45]:
X_train = X_train.select_dtypes(include=[np.number, 'bool', 'category'])
X_test = X_test.select_dtypes(include=[np.number, 'bool', 'category'])

In [47]:
for col in X_train.select_dtypes(include=['float64']).columns:
    X_train[col] = X_train[col].astype('float32')
    X_test[col] = X_test[col].astype('float32')

for col in X_train.select_dtypes(include=['int64']).columns:
    X_train[col] = pd.to_numeric(X_train[col], downcast='integer')
    X_test[col] = pd.to_numeric(X_test[col], downcast='integer')

In [48]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import lightgbm as lgb

n_split = 5

In [49]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [50]:
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
tpr_at_10_fpr_scores = []

In [51]:
print(f"Starting {n_splits}-Fold Cross-Validation...")

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'n_jobs': -1,
    'random_state': 42,
    'verbose': -1}

Starting 5-Fold Cross-Validation...


In [52]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y)):

    text_tr, text_val = train_tfidf[train_idx], train_tfidf[val_idx]
    
    dtrain = lgb.Dataset(X_tr, label=y_tr)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    model = lgb.train(lgb_params,dtrain,num_boost_round=1000,
                      valid_sets=[dtrain, dval],valid_names=['train', 'valid'],
                      callbacks=[lgb.early_stopping(stopping_rounds=50), 
                                 lgb.log_evaluation(period=100)])
    val_probs = model.predict(X_val)
    oof_preds[val_idx] = val_probs
    test_preds += model.predict(X_test) / n_splits
    #calculate TPR at 10% FPR
    fpr, tpr, thresholds = roc_curve(y_val, val_probs)
    idx = np.argmin(np.abs(fpr - 0.10))
    tpr_at_10 = tpr[idx]
    tpr_at_10_fpr_scores.append(tpr_at_10)
    print(f"Fold {fold + 1} | AUC: {roc_auc_score(y_val, val_probs):.4f} | TPR@10%FPR: {tpr_at_10:.4f}")

Training until validation scores don't improve for 50 rounds
[100]	train's auc: 0.900505	valid's auc: 0.897769
[200]	train's auc: 0.908544	valid's auc: 0.904149
[300]	train's auc: 0.912856	valid's auc: 0.906945
[400]	train's auc: 0.915531	valid's auc: 0.908637
[500]	train's auc: 0.91665	valid's auc: 0.90885
Early stopping, best iteration is:
[484]	train's auc: 0.916863	valid's auc: 0.909239
Fold 1 | AUC: 0.9092 | TPR@10%FPR: 0.7000
Training until validation scores don't improve for 50 rounds
[100]	train's auc: 0.900402	valid's auc: 0.898095
[200]	train's auc: 0.908663	valid's auc: 0.904423
[300]	train's auc: 0.912888	valid's auc: 0.907177
[400]	train's auc: 0.915673	valid's auc: 0.908775
[500]	train's auc: 0.917678	valid's auc: 0.909871
[600]	train's auc: 0.919174	valid's auc: 0.910493
[700]	train's auc: 0.920633	valid's auc: 0.911145
[800]	train's auc: 0.921834	valid's auc: 0.911563
[900]	train's auc: 0.922338	valid's auc: 0.911266
Early stopping, best iteration is:
[854]	train's auc:

In [53]:
print("\n--- Final Results ---")
print(f"Mean TPR at 10% FPR: {np.mean(tpr_at_10_fpr_scores):.4f} (+/- {np.std(tpr_at_10_fpr_scores):.4f})")


--- Final Results ---
Mean TPR at 10% FPR: 0.7054 (+/- 0.0049)


In [54]:
from sklearn.metrics import roc_curve

# Use the OOF predictions and true labels to find the threshold at 9.5% FPR
fpr, tpr, thresholds = roc_curve(y, oof_preds)

# Find the index where FPR is closest to 0.095
idx = np.argmin(np.abs(fpr - 0.95))
best_threshold = thresholds[idx]

print(f"Optimal Threshold at 10% FPR: {best_threshold:.4f}")

Optimal Threshold at 10% FPR: 0.2255


In [55]:
# Convert probabilities to binary 0/1 based on the threshold
final_test_binary = (test_preds >= best_threshold).astype(int)

# Check the distribution (How many 'useful' reviews did we find?)
print(pd.Series(final_test_binary).value_counts())

0    578072
1    121928
Name: count, dtype: int64


In [56]:
submission = pd.DataFrame(final_test_binary)

submission.to_csv('submission_predictions.csv', index=False, header=False)

print("Submission file 'submission_predictions.csv' has been created successfully!")

Submission file 'submission_predictions.csv' has been created successfully!
